# Notebook 1: KFC Reddit Data Collection

## Purpose
Collect posts and comments from the **r/kfc** subreddit using PRAW (Python Reddit API Wrapper).  
This mirrors the methodology used in the McDonald's social media analysis, adapted for KFC.

## What this notebook does
1. Connects to Reddit's API using your credentials
2. Scrapes posts from r/kfc via three endpoints: `.hot()`, `.new()`, `.top()`
3. For each post, extracts ALL comments (expanding nested replies)
4. Filters out `[removed]` and `[deleted]` comments
5. Filters comments to a 1-year date range (27 Mar 2025 – 27 Mar 2026)
6. Removes duplicate post-comment pairs using a unique key
7. Saves the final dataset as a CSV file

## Output
`kfc_comments_2025_2026.csv` — Columns: Post Title, Comment, Upvotes, Timestamp

## Reddit API Setup (one-time)
1. Go to https://www.reddit.com/prefs/apps/
2. Click **"create another app..."**
3. Select **script**, set redirect URI to `http://localhost:8080`
4. Copy the **client_id** and **client_secret**
5. Paste them in the configuration cell below

## Step 0 — Install & Import Libraries

In [ ]:
# Uncomment the line below if you need to install praw
# !pip install praw pandas

import praw
import pandas as pd
from datetime import datetime, timezone
import time

## Step 1 — Configuration

⚠️ **Replace the placeholder strings below with your actual Reddit API credentials.**

In [ ]:
# ============================================================
# FILL IN YOUR CREDENTIALS HERE
# ============================================================
REDDIT_CLIENT_ID     = "YOUR_CLIENT_ID"
REDDIT_CLIENT_SECRET = "YOUR_CLIENT_SECRET"
REDDIT_USER_AGENT    = "KFC_Scraper/1.0 by YOUR_REDDIT_USERNAME"
REDDIT_USERNAME      = ""   # optional — leave blank for read-only
REDDIT_PASSWORD      = ""   # optional — leave blank for read-only

SUBREDDIT_NAME = "kfc"

# Date range: 1 full year
START_DATE = datetime(2025, 3, 27, tzinfo=timezone.utc)
END_DATE   = datetime(2026, 3, 27, tzinfo=timezone.utc)

POST_LIMIT  = 1000
OUTPUT_FILE = "kfc_comments_2025_2026.csv"

print(f"Target: r/{SUBREDDIT_NAME}")
print(f"Date range: {START_DATE.date()} to {END_DATE.date()}")

## Step 2 — Connect to Reddit API

Creates a PRAW Reddit instance — the gateway that lets our script talk to Reddit.

In [ ]:
reddit = praw.Reddit(
    client_id=REDDIT_CLIENT_ID,
    client_secret=REDDIT_CLIENT_SECRET,
    user_agent=REDDIT_USER_AGENT,
    username=REDDIT_USERNAME if REDDIT_USERNAME else None,
    password=REDDIT_PASSWORD if REDDIT_PASSWORD else None,
)

if REDDIT_USERNAME:
    print(f"Authenticated as: {reddit.user.me()}")
else:
    print("Connected in read-only mode")

subreddit = reddit.subreddit(SUBREDDIT_NAME)
print(f"Subreddit loaded: r/{subreddit.display_name}")

## Step 3 — Define Helper Functions

- **`is_within_date_range()`** — checks if a Reddit timestamp is within our 1-year window
- **`extract_comments_from_post()`** — expands all nested replies, skips [removed]/[deleted]
- **`scrape_endpoint()`** — iterates through posts from a Reddit listing

In [ ]:
def is_within_date_range(utc_timestamp):
    """Check whether a Unix timestamp falls within START_DATE and END_DATE."""
    dt = datetime.fromtimestamp(utc_timestamp, tz=timezone.utc)
    return START_DATE <= dt <= END_DATE


def extract_comments_from_post(post):
    """
    Expand ALL nested comment replies from a post.
    replace_more(limit=0) loads all 'More Comments' links.
    Skips [removed] and [deleted] entries.
    """
    comments_data = []
    post.comments.replace_more(limit=0)
    
    for comment in post.comments.list():
        if comment.body in ("[removed]", "[deleted]"):
            continue
        if not is_within_date_range(comment.created_utc):
            continue
        
        comments_data.append({
            "Post Title": post.title,
            "Comment":    comment.body,
            "Upvotes":    comment.score,
            "Timestamp":  datetime.fromtimestamp(
                comment.created_utc, tz=timezone.utc
            ).strftime("%Y-%m-%d %H:%M:%S"),
        })
    return comments_data


def scrape_endpoint(name, generator, limit):
    """Scrape posts from one Reddit endpoint (.hot / .new / .top)."""
    print(f"\n--- Scraping .{name}() (up to {limit} posts) ---")
    all_comments, post_ids, count = [], set(), 0
    
    for post in generator(limit=limit):
        if not is_within_date_range(post.created_utc):
            continue
        post_ids.add(post.id)
        count += 1
        all_comments.extend(extract_comments_from_post(post))
        if count % 25 == 0:
            print(f"  ... {count} posts | {len(all_comments)} comments")
        time.sleep(0.5)
    
    print(f"  Done: {count} posts, {len(all_comments)} comments")
    return post_ids, all_comments

print("Helper functions defined.")

## Step 4 — Scrape All Three Endpoints

We collect from three endpoints to get a balanced dataset:
- **`.hot()`** — currently trending posts
- **`.new()`** — most recent posts
- **`.top()`** — highest-voted posts from the past year

In [ ]:
all_comments = []
all_post_ids = set()

# Endpoint 1: Hot (trending)
ids, comments = scrape_endpoint("hot", subreddit.hot, POST_LIMIT)
all_post_ids.update(ids); all_comments.extend(comments)

# Endpoint 2: New (most recent)
ids, comments = scrape_endpoint("new", subreddit.new, POST_LIMIT)
all_post_ids.update(ids); all_comments.extend(comments)

# Endpoint 3: Top (highest voted, past year)
ids, comments = scrape_endpoint(
    "top", lambda limit: subreddit.top(time_filter="year", limit=limit), POST_LIMIT
)
all_post_ids.update(ids); all_comments.extend(comments)

print(f"\nTotal raw comments: {len(all_comments)}")
print(f"Unique posts scraped: {len(all_post_ids)}")

## Step 5 — Deduplicate and Save to CSV

The same post can appear in multiple endpoints. We create a composite key from  
Post Title + Comment text to identify and remove exact duplicates.

In [ ]:
df = pd.DataFrame(all_comments)

if df.empty:
    print("ERROR: No comments found. Check your credentials and date range.")
else:
    # Deduplicate using a unique post-comment key
    df["unique_key"] = df["Post Title"] + " ||| " + df["Comment"]
    before = len(df)
    df.drop_duplicates(subset="unique_key", inplace=True)
    after = len(df)
    df.drop(columns=["unique_key"], inplace=True)
    
    print(f"Duplicates removed: {before - after}")
    print(f"Final dataset size: {after} comments")
    
    # Save
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f"\nSaved to: {OUTPUT_FILE}")
    
    # Summary
    print(f"\nTotal comments:  {len(df)}")
    print(f"Earliest entry:  {df['Timestamp'].min()}")
    print(f"Latest entry:    {df['Timestamp'].max()}")
    print(f"Avg upvotes:     {df['Upvotes'].mean():.1f}")
    df.head()